# Clase 02 — Estadística descriptiva

**Curso:** Estadística Aplicada con Python · **Bloque 1:** Describir los datos
**Tiempo estimado:** 100 minutos · **Capítulo del manual:** `manual/02_estadistica_descriptiva.md`

## Qué vas a ver en esta clase

- Media, mediana y moda: qué mide cada una y cuándo engañan.
- Dispersión: varianza, desviación estándar (poblacional frente a muestral) y rango intercuartílico (IQR).
- Cuantiles y percentiles, y la asimetría.
- Valores atípicos (*outliers*) con la regla del IQR y con el z-score (aquí solo como estandarización).
- Cómo elegir el resumen adecuado cuando hay valores extremos.

## Requisitos

- Clase 01 (muestra, población, estadístico, tipos de variables) y NumPy/Pandas básicos.
- Librerías: `numpy`, `pandas`, `matplotlib` (vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento, escribe (o piensa) qué esperas que ocurra; después compara con lo que ves.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio.

In [ ]:
# En Google Colab estas librerías ya vienen instaladas. En local: pip install numpy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. Nuestros datos: sueldos sintéticos con valores extremos

Generamos 300 sueldos mensuales (en €) con una distribución asimétrica, y **añadimos a mano cuatro valores muy altos**. Uno de ellos (250 000 €) simula un error de tecleo: alguien escribió dos ceros de más en un sueldo de 2 500 €. *(Datos sintéticos: ejemplo ilustrativo.)*

In [ ]:
sueldos_base = np.round(rng.lognormal(mean=7.6, sigma=0.35, size=300), 2)   # sueldos "normales" (asimétricos)
insertados = np.array([7_800.0, 9_500.0, 14_000.0, 250_000.0])                # valores extremos añadidos a mano

sueldos = pd.Series(np.concatenate([sueldos_base, insertados]), name="sueldo")
print(f"n = {len(sueldos)}")
print(sueldos.head(8).to_string())

plt.hist(sueldos_base, bins=40, color="steelblue", edgecolor="white")
plt.title("Sueldos generados (sin los valores insertados)")
plt.xlabel("Sueldo mensual (€)"); plt.ylabel("Empleados")
plt.show()

**Observa:** la mayoría cobra entre 1 500 y 2 700 €, con una cola hacia la derecha. Este histograma es solo un apoyo visual; en la clase 3 veremos con detalle cómo mirar una distribución.

---
## 2. Tendencia central: media, mediana y moda

Calculamos los tres resúmenes sobre los 304 sueldos (los 300 normales más los 4 insertados).

In [ ]:
print(f"Media   : {sueldos.mean():>10,.1f} €")
print(f"Mediana : {sueldos.median():>10,.1f} €")
print(f"Media de los 300 sueldos sin insertar : {sueldos_base.mean():,.1f} €")
print(f"Mediana de los 300 sueldos sin insertar: {np.median(sueldos_base):,.1f} €")

**¿Qué observas?** Compara: al añadir solo 4 valores, ¿cuánto se ha movido la media? ¿Y la mediana? Anótalo antes de seguir.

*(Tu respuesta aquí.)*

### La moda con datos continuos

Los sueldos tienen céntimos, así que casi ningún valor se repite. Veamos qué significa entonces la "moda".

In [ ]:
print("Valores distintos:", sueldos.nunique(), "de", len(sueldos))
print("Veces que se repite el valor más frecuente:", sueldos.value_counts().max())

# Solución razonable: agrupar en intervalos de 500 € y buscar el intervalo más frecuente
intervalos = (sueldos // 500 * 500).astype(int)
print(intervalos.value_counts().sort_index().to_string())
print("Intervalo modal (empieza en):", intervalos.mode()[0], "€")

**Observa:** con los datos "crudos" la moda no significa nada (todos los valores aparecen una vez). Agrupando en intervalos sí obtenemos una idea de dónde se concentran los sueldos, y eso ya es casi un histograma.

---
## 3. Dispersión: varianza, desviación estándar e IQR

Medimos cuánto se separan los sueldos. Comparamos una medida que usa todos los valores (desviación estándar) con una que usa solo el "medio 50 %" (IQR).

In [ ]:
def iqr(x):
    q1, q3 = np.quantile(x, [0.25, 0.75])
    return q3 - q1

print(f"Rango                  : {sueldos.max() - sueldos.min():>12,.1f} €")
print(f"Varianza muestral      : {sueldos.var():>12,.1f} €²")
print(f"Desviación estándar    : {sueldos.std():>12,.1f} €")
print(f"IQR                     : {iqr(sueldos):>12,.1f} €")
print()
print(f"Sin los 4 insertados -> desviación estándar: {sueldos_base.std(ddof=1):,.1f} €   IQR: {iqr(sueldos_base):,.1f} €")

**¿Qué observas?** ¿Cuál de las dos medidas de dispersión ha cambiado más al añadir los 4 valores extremos? ¿Por qué?

*(Tu respuesta aquí.)*

### Desviación estándar poblacional frente a muestral (`ddof`)

`numpy` y `pandas` calculan la desviación estándar con **divisores distintos** por defecto. Lo comprobamos con un ejemplo que puedes verificar a mano: `[2, 4, 6]`. La media es 4, las desviaciones son −2, 0, 2, la suma de cuadrados es 8.

In [ ]:
mini = [2, 4, 6]
suma_cuadrados = sum((v - np.mean(mini)) ** 2 for v in mini)      # = 8

print("Suma de cuadrados            :", suma_cuadrados)
print("A mano, dividiendo por n     :", round(np.sqrt(suma_cuadrados / 3), 3), "  (ddof=0)")
print("A mano, dividiendo por n - 1 :", round(np.sqrt(suma_cuadrados / 2), 3), "  (ddof=1)")
print()
print("np.std(mini)               :", round(np.std(mini), 3), "  <- ddof=0 por defecto")
print("pd.Series(mini).std()      :", round(pd.Series(mini).std(), 3), "  <- ddof=1 por defecto")
print("np.std(mini, ddof=1)       :", round(np.std(mini, ddof=1), 3))
print()
# ¿Importa con muchos datos?
print(f"Con los 304 sueldos: np.std = {np.std(sueldos):.2f}   pandas std = {sueldos.std():.2f}   (casi iguales)")

**¿Qué observas?** Con 3 datos, la diferencia es enorme (1,633 frente a 2). Con 304 datos casi desaparece. Cuando analices una muestra pequeña, escribe siempre `ddof=1` de forma explícita.

---
## 4. Cuantiles y percentiles

Los cuantiles dividen los datos ordenados. `pandas` usa por defecto interpolación lineal. Primero comprobamos a mano con los 9 sueldos del capítulo, donde las posiciones son exactas (0,25 · 8 = 2, 0,5 · 8 = 4, 0,75 · 8 = 6).

In [ ]:
nueve = pd.Series([1600, 1800, 1900, 2000, 2100, 2200, 2500, 2800, 9500])
print(nueve.quantile([0.25, 0.5, 0.75]).to_string())
print("Tercer, quinto y séptimo dato ordenados:", nueve.sort_values().iloc[[2, 4, 6]].tolist())

In [ ]:
percentiles = sueldos.quantile([0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(1)
percentiles.index = [f"P{int(p * 100)}" for p in percentiles.index]
print(percentiles.to_string())
print()
print(sueldos.describe().round(1).to_string())

**Observa:** `describe()` da de una vez n, media, desviación estándar, mínimo, cuartiles y máximo. Compara la media (`mean`) con la mediana (`50%`) y el máximo con el percentil 75: ya se intuye que hay asimetría y valores extremos. Lo que `describe()` **no** hace es marcar los atípicos ni medir la asimetría.

---
## 5. Asimetría

Comparamos los sueldos con una muestra simétrica de referencia (alturas sintéticas en cm, también *ejemplo ilustrativo*). Miramos la diferencia media − mediana, el coeficiente `skew()` y un histograma.

In [ ]:
alturas = pd.Series(rng.normal(170, 8, size=304), name="altura")     # muestra simétrica de referencia

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for eje, serie, titulo, unidad in [(ejes[0], sueldos, "Sueldos (asimétrica)", "€"),
                                   (ejes[1], alturas, "Alturas (simétrica)", "cm")]:
    eje.hist(serie, bins=50, color="lightsteelblue", edgecolor="white")
    eje.axvline(serie.mean(), color="crimson", linestyle="--", label=f"media = {serie.mean():,.0f}")
    eje.axvline(serie.median(), color="black", linestyle="-", label=f"mediana = {serie.median():,.0f}")
    eje.set_title(titulo); eje.set_xlabel(unidad); eje.legend()
plt.show()

print(f"Sueldos: media − mediana = {sueldos.mean() - sueldos.median():+.1f} €   skew = {sueldos.skew():.2f}")
print(f"Alturas: media − mediana = {alturas.mean() - alturas.median():+.2f} cm   skew = {alturas.skew():.2f}")

**¿Qué observas?** En la distribución simétrica, media y mediana casi coinciden y `skew()` está cerca de 0. En los sueldos, la media queda por encima de la mediana y `skew()` es claramente positivo: la cola larga a la derecha tira de la media.

---
## 6. Valores atípicos: regla del IQR y z-score

Aplicamos las dos reglas a los 304 sueldos. **Recuerda:** en esta clase el z-score es solo una **estandarización** (`(x − media) / desviación estándar`); su relación con la distribución normal se ve en la clase 5.

In [ ]:
q1, q3 = sueldos.quantile([0.25, 0.75])
lim_inf, lim_sup = q1 - 1.5 * iqr(sueldos), q3 + 1.5 * iqr(sueldos)
atipicos_iqr = sueldos[(sueldos < lim_inf) | (sueldos > lim_sup)]

z = (sueldos - sueldos.mean()) / sueldos.std()          # estandarización: media 0, desviación estándar 1
atipicos_z = sueldos[z.abs() > 3]

print(f"Límites de la regla del IQR: [{lim_inf:,.0f} €, {lim_sup:,.0f} €]")
print(f"Atípicos por IQR    : {len(atipicos_iqr)}")
print(f"Atípicos por |z| > 3: {len(atipicos_z)} -> {sorted(atipicos_z.round(0).tolist())}")
print(f"z-score media = {z.mean():.2f}, desviación estándar = {z.std():.2f}")
print("Los 4 sueldos insertados a mano, ¿detectados por el IQR?", [bool(v in set(atipicos_iqr)) for v in insertados])
print("Los 4 sueldos insertados a mano, ¿detectados por z?     ", [bool(v in set(atipicos_z)) for v in insertados])

**¿Qué observas?** La regla del IQR marca bastantes más valores que |z| > 3. ¿Por qué? Piensa en qué le pasa a la media y a la desviación estándar cuando hay un valor de 250 000 €.

*(Tu respuesta aquí.)*

### El z-score con pocos datos: el ejemplo de 9 sueldos

In [ ]:
z_nueve = (nueve - nueve.mean()) / nueve.std()
print("z-score:", z_nueve.round(2).tolist())
print("z máximo posible con n = 9:", round((len(nueve) - 1) / np.sqrt(len(nueve)), 2))

q1n, q3n = nueve.quantile([0.25, 0.75])
print("Límites IQR:", (q1n - 1.5 * (q3n - q1n), q3n + 1.5 * (q3n - q1n)), "->", nueve[(nueve < q1n - 1.5 * (q3n - q1n)) | (nueve > q3n + 1.5 * (q3n - q1n))].tolist())

**Observa:** el 9 500 tiene z = 2,64: no supera 3, así que |z| > 3 **no lo detecta**, mientras que la regla del IQR sí. Con *n* = 9 es imposible superar z = 2,67. El propio atípico contamina la media y la desviación estándar.

---
## 7. Experimento guiado: añadir valores extremos uno a uno

Partimos de los 300 sueldos "normales" y añadimos progresivamente valores cada vez más extremos: 8 000, 15 000, 30 000, 60 000, 120 000 y 250 000 €. En cada paso calculamos dos resúmenes de centro (media y mediana) y dos de dispersión (desviación estándar e IQR). ¿Cuáles se mueven y cuáles no?

**Antes de ejecutar, predice:** ¿qué pasará con la media tras añadir el sexto valor? ¿Y con la mediana? ¿Y con el IQR?

In [ ]:
extremos = [8_000, 15_000, 30_000, 60_000, 120_000, 250_000]

filas = []
for k in range(len(extremos) + 1):
    muestra_k = np.concatenate([sueldos_base, extremos[:k]])        # k = número de extremos añadidos
    filas.append({"k": k,
                  "media": muestra_k.mean(),
                  "mediana": np.median(muestra_k),
                  "desv_std": muestra_k.std(ddof=1),
                  "iqr": iqr(muestra_k)})

evolucion = pd.DataFrame(filas).set_index("k").round(1)
display(evolucion)

cambio = (evolucion.loc[len(extremos)] / evolucion.loc[0] - 1)
print("Cambio relativo entre 0 y 6 extremos añadidos:")
print(cambio.map("{:+.0%}".format).to_string())

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(evolucion.index, evolucion["media"], "o-", color="crimson", label="media")
ejes[0].plot(evolucion.index, evolucion["mediana"], "s-", color="black", label="mediana")
ejes[0].set_title("Centro"); ejes[0].set_xlabel("Valores extremos añadidos"); ejes[0].set_ylabel("€"); ejes[0].legend()
ejes[1].plot(evolucion.index, evolucion["desv_std"], "o-", color="crimson", label="desviación estándar")
ejes[1].plot(evolucion.index, evolucion["iqr"], "s-", color="black", label="IQR")
ejes[1].set_title("Dispersión"); ejes[1].set_xlabel("Valores extremos añadidos"); ejes[1].set_ylabel("€"); ejes[1].legend()
plt.show()

**¿Qué observas?** Anota tus conclusiones:

- ¿Cuánto ha crecido la media y cuánto la mediana entre k = 0 y k = 6? ¿Coincide con tu predicción?
- ¿Cuál de las dos medidas de dispersión se dispara y cuál casi no se mueve?
- Si tuvieras que informar del "sueldo típico" y de su dispersión después de añadir los 6 extremos, ¿qué dos cifras darías?

*(Tu respuesta aquí.)*

**Conclusión:** la media y la desviación estándar usan todos los valores y se dejan arrastrar por los extremos; la mediana y el IQR dependen del orden de los datos y apenas se inmutan. Cuando hay valores extremos, los resúmenes robustos cuentan mejor la historia del grueso de los datos.

---
## 8. ¿Y qué hacemos con los atípicos? Investigar antes de eliminar

Detectar un atípico no autoriza a borrarlo. Miramos los valores marcados por la regla del IQR que están por encima de 7 000 € y comparamos tres versiones de los datos.

In [ ]:
print("Valores muy altos marcados por el IQR:", sorted(atipicos_iqr[atipicos_iqr > 7_000].tolist()))

# Suponemos que hemos comprobado con RR. HH.: 250 000 era un error de tecleo (debía ser 2 500); los otros tres son sueldos reales.
sueldos_corregidos = sueldos.replace(250_000.0, 2_500.0)

def resumen(x):
    """Resumen robusto de una muestra numérica."""
    return pd.Series({"n": len(x), "media": x.mean(), "mediana": x.median(),
                      "desv_std": x.std(ddof=1), "iqr": iqr(x), "asimetria": x.skew()})

comparacion = pd.DataFrame({
    "con el error": resumen(sueldos),
    "error corregido": resumen(sueldos_corregidos),
    "sin los 4 insertados": resumen(pd.Series(sueldos_base)),
}).round(1)
display(comparacion)

**¿Qué observas?** Corregir el error de tecleo (un dato verificado) cambia mucho la media y la desviación estándar; los tres sueldos reales muy altos siguen tirando de la media hacia arriba, pero no se deben borrar: son personas reales. La mediana y el IQR son casi iguales en las tres versiones.

> **Regla práctica:** corrige lo que puedas verificar como error; conserva lo real y elige resúmenes robustos; y documenta siempre qué has hecho.

---
## Resumen de la clase

1. **Centro:** media (usa todo, sensible a extremos), mediana (robusta), moda (solo con categorías o datos discretos).
2. **Dispersión:** desviación estándar (sensible) e IQR (robusto). `np.std` usa `ddof=0`, `pandas` usa `ddof=1`: decide y escríbelo.
3. **Cuantiles y percentiles:** el cuantil *p* deja una fracción *p* de los datos por debajo; `describe()` es un buen primer vistazo.
4. **Asimetría:** si media > mediana y `skew()` > 0, hay cola a la derecha.
5. **Atípicos:** regla del IQR (robusta) y z-score (solo estandarización aquí; con pocos datos puede no detectar). Investiga antes de eliminar.
6. **Elegir el resumen:** simétrico sin extremos -> media y desviación estándar; asimétrico o con extremos -> mediana e IQR. Y siempre, mira los datos.

**Siguiente paso:** Clase 3 — Distribuciones empíricas (histogramas, ECDF y más formas de mirar una distribución). Y ahora, a practicar: `notebooks_practica/clase02_practica_estadistica_descriptiva.ipynb`.